# Proyecto 3 — Optimización de Pricing en Rutas de Ferry

## Notebook 6 — Recomendador formal de pricing

En este notebook construimos un **motor formal de recomendación de pricing** para **Levante Ferries**.

El objetivo es convertir los notebooks anteriores en una salida accionable:

- Qué precio recomendar.
- Qué acción aplicar.
- Qué impacto económico esperar.
- Qué nivel de confianza tiene la recomendación.
- Qué explicación de negocio acompaña a cada decisión.

## 1. Objetivo del notebook

Este notebook responde a una pregunta clave:

> Con la demanda, elasticidad, margen, ocupación y competencia que tenemos, ¿qué acción de pricing tiene más sentido?

El recomendador no es un piloto automático. Es una herramienta de apoyo para revenue management.

In [ ]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import json
import warnings
import joblib

from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")

print("Librerías importadas correctamente.")

In [ ]:
# ============================================================
# 2. CONEXIÓN CON GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive conectado correctamente.")
except:
    print("No estás ejecutando este notebook en Google Colab o Drive ya está montado.")

In [ ]:
# ============================================================
# 3. DEFINICIÓN DE RUTAS DEL PROYECTO
# ============================================================

base_path = "/content/drive/MyDrive/7 Colab Notebooks/1 Porfolio/Balearia/3 Pricing"

data_path = f"{base_path}/data/processed/ferry_pricing_dataset.csv"
reports_path = f"{base_path}/reports"
images_path = f"{base_path}/images"
models_path = f"{base_path}/models"
dashboard_path = f"{base_path}/dashboard"

os.makedirs(reports_path, exist_ok=True)
os.makedirs(images_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(dashboard_path, exist_ok=True)

elasticity_path = f"{reports_path}/elasticity_comparison_by_route.csv"

print("Ruta base:")
print(base_path)

## 2. Funciones auxiliares

Creamos funciones de limpieza, exportación, scoring, clasificación de riesgos y explicación de recomendaciones.

In [ ]:
# ============================================================
# 4. FUNCIONES DE LIMPIEZA Y EXPORTACIÓN
# ============================================================

def force_numeric_columns(dataframe, columns):
    df_copy = dataframe.copy()

    for col in columns:
        if col not in df_copy.columns:
            continue

        direct = pd.to_numeric(df_copy[col], errors="coerce")

        if direct.notna().mean() >= 0.80:
            df_copy[col] = direct
        else:
            df_copy[col] = (
                df_copy[col].astype(str)
                .str.replace("€", "", regex=False)
                .str.replace("%", "", regex=False)
                .str.replace(" ", "", regex=False)
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
            )
            df_copy[col] = pd.to_numeric(df_copy[col], errors="coerce")

    return df_copy


def export_table_files(dataframe, output_folder, file_name, sheet_name="Data"):
    os.makedirs(output_folder, exist_ok=True)

    csv_path = f"{output_folder}/{file_name}.csv"
    csv_excel_es_path = f"{output_folder}/{file_name}_excel_es.csv"
    xlsx_path = f"{output_folder}/{file_name}.xlsx"

    df_export = dataframe.copy()

    for col in df_export.columns:
        if pd.api.types.is_datetime64_any_dtype(df_export[col]):
            df_export[col] = df_export[col].dt.strftime("%d/%m/%Y %H:%M")

    dataframe.to_csv(csv_path, index=False, encoding="utf-8-sig")

    df_export.to_csv(
        csv_excel_es_path,
        index=False,
        sep=";",
        decimal=",",
        encoding="utf-8-sig"
    )

    safe_sheet_name = sheet_name[:31]

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        dataframe.to_excel(writer, index=False, sheet_name=safe_sheet_name)
        ws = writer.sheets[safe_sheet_name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for cell in ws[1]:
            cell.font = Font(bold=True)

        for col_idx, col_name in enumerate(dataframe.columns, start=1):
            letter = get_column_letter(col_idx)

            if pd.api.types.is_float_dtype(dataframe[col_name]):
                if any(token in col_name for token in ["pct", "rate", "score", "share"]):
                    number_format = "0.00%"
                else:
                    number_format = "0.00"
                for cell in ws[letter][1:]:
                    cell.number_format = number_format

            elif pd.api.types.is_integer_dtype(dataframe[col_name]):
                for cell in ws[letter][1:]:
                    cell.number_format = "0"

        for column_cells in ws.columns:
            max_length = 0
            letter = get_column_letter(column_cells[0].column)
            for cell in column_cells:
                if cell.value is not None:
                    max_length = max(max_length, len(str(cell.value)))
            ws.column_dimensions[letter].width = min(max_length + 2, 45)

    print(f"Exportado: {file_name}")


print("Funciones de limpieza y exportación creadas.")

In [ ]:
# ============================================================
# 5. FUNCIONES DE SCORING Y RECOMENDACIÓN
# ============================================================

def find_latest_model(models_folder):
    model_files = glob.glob(f"{models_folder}/demand_prediction_model_*.joblib")
    if len(model_files) == 0:
        return None
    return max(model_files, key=os.path.getmtime)


def clip_between(value, lower=0, upper=1):
    return min(max(value, lower), upper)


def classify_occupancy(occupancy):
    if occupancy >= 0.95:
        return "Saturation risk"
    elif occupancy >= 0.85:
        return "High occupancy"
    elif occupancy >= 0.65:
        return "Healthy occupancy"
    elif occupancy >= 0.50:
        return "Soft demand"
    else:
        return "Low occupancy risk"


def classify_margin(margin_pct):
    if margin_pct >= 0.35:
        return "Very strong margin"
    elif margin_pct >= 0.25:
        return "Healthy margin"
    elif margin_pct >= 0.15:
        return "Weak margin"
    else:
        return "Margin pressure"


def classify_competition(price_index):
    if price_index >= 1.12:
        return "Premium risk"
    elif price_index >= 1.03:
        return "Slightly premium"
    elif price_index >= 0.97:
        return "Market aligned"
    elif price_index >= 0.90:
        return "Slightly discounted"
    else:
        return "Deep discount"


def classify_elasticity(elasticity):
    if elasticity > -0.50:
        return "Very inelastic"
    elif elasticity > -1.00:
        return "Inelastic"
    elif elasticity >= -1.20:
        return "Unit elasticity area"
    else:
        return "Elastic"


def calculate_recommendation_score(row):
    revenue_component = clip_between((row["revenue_uplift_pct"] + 0.10) / 0.30)
    margin_component = clip_between((row["margin_uplift_pct"] + 0.10) / 0.35)

    occupancy = row["recommended_occupancy_rate"]
    occupancy_component = 1 - min(abs(occupancy - 0.80) / 0.45, 1)

    price_index = row["recommended_price_index"]

    if 0.97 <= price_index <= 1.08:
        competitor_component = 1.0
    elif 0.90 <= price_index < 0.97 or 1.08 < price_index <= 1.15:
        competitor_component = 0.65
    else:
        competitor_component = 0.30

    penalty = 0

    if occupancy < 0.50:
        penalty += 0.25
    if occupancy > 0.96:
        penalty += 0.12
    if row["recommended_margin_pct"] < 0.15:
        penalty += 0.25
    if price_index > 1.18:
        penalty += 0.18

    score = (
        revenue_component * 0.30 +
        margin_component * 0.35 +
        occupancy_component * 0.20 +
        competitor_component * 0.15 -
        penalty
    )

    return clip_between(score)


def classify_confidence(score):
    if score >= 0.78:
        return "High"
    elif score >= 0.58:
        return "Medium"
    elif score >= 0.40:
        return "Low"
    else:
        return "Manual review"


def assign_pricing_action(row):
    price_change = row["recommended_price_change_pct"]
    occupancy = row["recommended_occupancy_rate"]
    margin_pct = row["recommended_margin_pct"]
    elasticity = row["elasticity_for_simulation"]
    price_index = row["recommended_price_index"]
    score = row["recommendation_score"]

    if score < 0.40:
        return "Manual review"

    if (
        price_change > 0 and
        occupancy >= 0.72 and
        margin_pct >= 0.22 and
        elasticity > -1.10 and
        price_index <= 1.15
    ):
        return "Increase price"

    if (
        price_change < 0 and
        occupancy < 0.68 and
        elasticity <= -0.90 and
        margin_pct >= 0.10
    ):
        return "Promotional action"

    if margin_pct < 0.15:
        return "Margin review"

    if occupancy >= 0.95:
        return "Capacity / premium review"

    return "Maintain / monitor"


def explain_recommendation(row):
    reasons = []

    if row["recommended_price_change_pct"] > 0:
        reasons.append("propone una subida controlada de precio")
    elif row["recommended_price_change_pct"] < 0:
        reasons.append("propone un ajuste promocional")
    else:
        reasons.append("mantiene el precio actual como opción equilibrada")

    if row["revenue_uplift_pct"] > 0.02:
        reasons.append("mejora el revenue esperado")
    elif row["revenue_uplift_pct"] < -0.02:
        reasons.append("reduce el revenue esperado")

    if row["margin_uplift_pct"] > 0.02:
        reasons.append("mejora el margen esperado")
    elif row["margin_uplift_pct"] < -0.02:
        reasons.append("reduce el margen esperado")

    if row["recommended_occupancy_rate"] < 0.55:
        reasons.append("presenta riesgo de baja ocupación")
    elif row["recommended_occupancy_rate"] > 0.95:
        reasons.append("presenta riesgo de saturación")
    else:
        reasons.append("mantiene una ocupación razonable")

    if row["recommended_price_index"] > 1.15:
        reasons.append("queda muy por encima del competidor")
    elif row["recommended_price_index"] < 0.90:
        reasons.append("queda claramente por debajo del competidor")
    else:
        reasons.append("mantiene una posición competitiva razonable")

    if row["elasticity_for_simulation"] > -1:
        reasons.append("la ruta muestra sensibilidad al precio moderada")
    else:
        reasons.append("la ruta muestra sensibilidad relevante al precio")

    return "Se recomienda esta acción porque " + "; ".join(reasons) + "."


print("Funciones de scoring creadas.")

## 3. Carga del dataset base

Cargamos el dataset principal del Proyecto 3.

In [ ]:
# ============================================================
# 6. CARGA DEL DATASET
# ============================================================

df = pd.read_csv(data_path, parse_dates=["trip_date", "departure_datetime"])

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

df.head()

In [ ]:
# ============================================================
# 7. REVISIÓN DE COLUMNAS NUMÉRICAS
# ============================================================

numeric_columns_to_check = [
    "capacity",
    "base_price",
    "avg_ticket_price",
    "competitor_price",
    "price_index_vs_competitor",
    "route_elasticity",
    "days_before_departure",
    "weather_score",
    "event_flag",
    "expected_demand",
    "tickets_sold",
    "occupancy_rate",
    "revenue",
    "fixed_operational_cost",
    "variable_cost_per_passenger",
    "total_operational_cost",
    "margin",
    "margin_pct"
]

df = force_numeric_columns(df, numeric_columns_to_check)

print("Columnas numéricas revisadas correctamente.")
df[numeric_columns_to_check].dtypes

## 4. Carga de elasticidad

Cargamos la elasticidad del Notebook 3.  
Si no existe el archivo, usamos la elasticidad simulada del dataset.

In [ ]:
# ============================================================
# 8. CARGA DE ELASTICIDAD
# ============================================================

if os.path.exists(elasticity_path):
    elasticity_df = pd.read_csv(elasticity_path)
    print("Elasticidad cargada desde Notebook 3.")
else:
    print("No se encontró archivo de elasticidad. Se usa route_elasticity del dataset.")
    elasticity_df = df[["route", "route_elasticity"]].drop_duplicates().copy()
    elasticity_df["elasticity_for_simulation"] = elasticity_df["route_elasticity"]

elasticity_df = force_numeric_columns(
    elasticity_df,
    [
        "simple_elasticity",
        "adjusted_elasticity",
        "simulated_route_elasticity",
        "elasticity_for_simulation",
        "avg_occupancy",
        "avg_margin_pct",
        "avg_price_index"
    ]
)

elasticity_df.head()

## 5. Demanda base

Intentamos cargar el modelo de demanda del Notebook 4.

Si existe, usamos demanda predicha.  
Si no existe, usamos `tickets_sold` como demanda base observada.

In [ ]:
# ============================================================
# 9. CARGA DEL MODELO DE DEMANDA
# ============================================================

latest_model_path = find_latest_model(models_path)

if latest_model_path is not None:
    demand_model = joblib.load(latest_model_path)
    model_available = True
    print("Modelo de demanda cargado:")
    print(latest_model_path)
else:
    demand_model = None
    model_available = False
    print("No se encontró modelo de demanda. Se usará demanda observada.")

In [ ]:
# ============================================================
# 10. PREPARAR DEMANDA BASE
# ============================================================

numeric_features = [
    "month",
    "week",
    "departure_hour",
    "is_weekend",
    "high_season_flag",
    "capacity",
    "base_price",
    "avg_ticket_price",
    "competitor_price",
    "price_index_vs_competitor",
    "route_elasticity",
    "days_before_departure",
    "weather_score",
    "event_flag"
]

categorical_features = [
    "route",
    "origin",
    "destination",
    "route_type",
    "vessel_type",
    "day_of_week",
    "season",
    "booking_window"
]

feature_columns = numeric_features + categorical_features

recommendation_base = df.copy()

missing_model_features = [col for col in feature_columns if col not in recommendation_base.columns]

if model_available and len(missing_model_features) == 0:
    X_all = recommendation_base[feature_columns].copy()
    predicted_demand = demand_model.predict(X_all)

    recommendation_base["predicted_base_demand"] = np.minimum(
        np.maximum(predicted_demand, 0),
        recommendation_base["capacity"]
    )
    recommendation_base["demand_source"] = "Demand model"
else:
    recommendation_base["predicted_base_demand"] = recommendation_base["tickets_sold"]
    recommendation_base["demand_source"] = "Observed tickets"

recommendation_base["predicted_occupancy_rate"] = (
    recommendation_base["predicted_base_demand"] / recommendation_base["capacity"]
)

elasticity_map = dict(
    zip(
        elasticity_df["route"],
        elasticity_df["elasticity_for_simulation"]
    )
)

recommendation_base["elasticity_for_simulation"] = recommendation_base["route"].map(elasticity_map)
recommendation_base["elasticity_for_simulation"] = recommendation_base["elasticity_for_simulation"].fillna(recommendation_base["route_elasticity"])
recommendation_base["elasticity_for_simulation"] = recommendation_base["elasticity_for_simulation"].fillna(-1.0)

recommendation_base[[
    "trip_id",
    "route",
    "avg_ticket_price",
    "tickets_sold",
    "predicted_base_demand",
    "predicted_occupancy_rate",
    "elasticity_for_simulation",
    "demand_source"
]].head()

## 6. Simulación de escenarios del recomendador

Generamos escenarios de precio desde -20% hasta +20%.

Después elegiremos el mejor escenario por viaje usando un score de negocio.

In [ ]:
# ============================================================
# 11. CONFIGURACIÓN DE ESCENARIOS
# ============================================================

price_scenarios = [-0.20, -0.15, -0.10, -0.05, 0.00, 0.05, 0.10, 0.15, 0.20]

scenario_config = pd.DataFrame({
    "price_change_pct": price_scenarios,
    "scenario_label": [f"{x:+.0%}" for x in price_scenarios]
})

scenario_config["scenario_type"] = np.where(
    scenario_config["price_change_pct"] < 0,
    "Discount",
    np.where(
        scenario_config["price_change_pct"] > 0,
        "Increase",
        "Current"
    )
)

scenario_config

In [ ]:
# ============================================================
# 12. GENERAR ESCENARIOS POR VIAJE
# ============================================================

scenario_frames = []

for _, scenario_row in scenario_config.iterrows():

    price_change = scenario_row["price_change_pct"]
    scenario_label = scenario_row["scenario_label"]
    scenario_type = scenario_row["scenario_type"]

    temp = recommendation_base.copy()

    temp["scenario_label"] = scenario_label
    temp["scenario_type"] = scenario_type
    temp["price_change_pct"] = price_change

    temp["recommended_ticket_price"] = temp["avg_ticket_price"] * (1 + price_change)

    temp["recommended_price_index"] = (
        temp["recommended_ticket_price"] / temp["competitor_price"]
    )

    temp["recommended_demand"] = (
        temp["predicted_base_demand"] *
        ((temp["recommended_ticket_price"] / temp["avg_ticket_price"]) ** temp["elasticity_for_simulation"])
    )

    temp["recommended_tickets_sold"] = np.minimum(
        np.maximum(temp["recommended_demand"], 0),
        temp["capacity"]
    )

    temp["recommended_occupancy_rate"] = (
        temp["recommended_tickets_sold"] / temp["capacity"]
    )

    temp["recommended_revenue"] = (
        temp["recommended_tickets_sold"] * temp["recommended_ticket_price"]
    )

    temp["recommended_operational_cost"] = (
        temp["fixed_operational_cost"] +
        temp["recommended_tickets_sold"] * temp["variable_cost_per_passenger"]
    )

    temp["recommended_margin"] = (
        temp["recommended_revenue"] - temp["recommended_operational_cost"]
    )

    temp["recommended_margin_pct"] = np.where(
        temp["recommended_revenue"] > 0,
        temp["recommended_margin"] / temp["recommended_revenue"],
        0
    )

    temp["revenue_uplift"] = temp["recommended_revenue"] - temp["revenue"]

    temp["revenue_uplift_pct"] = np.where(
        temp["revenue"] > 0,
        temp["revenue_uplift"] / temp["revenue"],
        0
    )

    temp["margin_uplift"] = temp["recommended_margin"] - temp["margin"]

    temp["margin_uplift_pct"] = np.where(
        temp["margin"] != 0,
        temp["margin_uplift"] / temp["margin"],
        0
    )

    scenario_frames.append(temp)

recommendation_scenarios = pd.concat(scenario_frames, ignore_index=True)

print(f"Escenarios generados: {recommendation_scenarios.shape[0]:,}")

recommendation_scenarios.head()

## 7. Scoring de escenarios

Cada escenario recibe una puntuación de 0 a 1 combinando:

- Revenue uplift.
- Margin uplift.
- Ocupación saludable.
- Posición competitiva.
- Penalización por riesgo.

In [ ]:
# ============================================================
# 13. CALCULAR SCORE DE RECOMENDACIÓN
# ============================================================

recommendation_scenarios["occupancy_class"] = recommendation_scenarios["recommended_occupancy_rate"].apply(classify_occupancy)
recommendation_scenarios["margin_class"] = recommendation_scenarios["recommended_margin_pct"].apply(classify_margin)
recommendation_scenarios["competitive_position"] = recommendation_scenarios["recommended_price_index"].apply(classify_competition)
recommendation_scenarios["elasticity_class"] = recommendation_scenarios["elasticity_for_simulation"].apply(classify_elasticity)

recommendation_scenarios["recommendation_score"] = recommendation_scenarios.apply(
    calculate_recommendation_score,
    axis=1
)

recommendation_scenarios["confidence_level"] = recommendation_scenarios["recommendation_score"].apply(classify_confidence)

recommendation_scenarios[[
    "trip_id",
    "route",
    "scenario_label",
    "recommendation_score",
    "confidence_level",
    "occupancy_class",
    "margin_class",
    "competitive_position"
]].head()

## 8. Mejor escenario por viaje

Para cada `trip_id`, seleccionamos el escenario con mayor score.

In [ ]:
# ============================================================
# 14. SELECCIÓN DEL MEJOR ESCENARIO POR VIAJE
# ============================================================

best_trip_recommendations = (
    recommendation_scenarios
    .sort_values(["trip_id", "recommendation_score"], ascending=[True, False])
    .groupby("trip_id")
    .head(1)
    .copy()
)

best_trip_recommendations["recommended_price_change_pct"] = best_trip_recommendations["price_change_pct"]

best_trip_recommendations["pricing_action"] = best_trip_recommendations.apply(
    assign_pricing_action,
    axis=1
)

best_trip_recommendations["recommendation_explanation"] = best_trip_recommendations.apply(
    explain_recommendation,
    axis=1
)

best_trip_recommendations = best_trip_recommendations[[
    "trip_id",
    "trip_date",
    "departure_datetime",
    "route",
    "origin",
    "destination",
    "season",
    "day_of_week",
    "departure_hour",
    "booking_window",
    "capacity",
    "avg_ticket_price",
    "recommended_ticket_price",
    "recommended_price_change_pct",
    "competitor_price",
    "recommended_price_index",
    "tickets_sold",
    "predicted_base_demand",
    "recommended_tickets_sold",
    "occupancy_rate",
    "predicted_occupancy_rate",
    "recommended_occupancy_rate",
    "revenue",
    "recommended_revenue",
    "revenue_uplift",
    "revenue_uplift_pct",
    "margin",
    "recommended_margin",
    "margin_uplift",
    "margin_uplift_pct",
    "margin_pct",
    "recommended_margin_pct",
    "elasticity_for_simulation",
    "elasticity_class",
    "occupancy_class",
    "margin_class",
    "competitive_position",
    "recommendation_score",
    "confidence_level",
    "pricing_action",
    "recommendation_explanation",
    "demand_source"
]].sort_values(["route", "trip_date", "departure_hour"])

best_trip_recommendations.head()

## 9. Resumen por ruta

Agregamos las recomendaciones para entender la estrategia sugerida por ruta.

In [ ]:
# ============================================================
# 15. RESUMEN POR RUTA
# ============================================================

route_recommendation_summary = (
    best_trip_recommendations
    .groupby("route")
    .agg(
        trips=("trip_id", "count"),
        avg_current_price=("avg_ticket_price", "mean"),
        avg_recommended_price=("recommended_ticket_price", "mean"),
        avg_price_change_pct=("recommended_price_change_pct", "mean"),
        avg_current_occupancy=("occupancy_rate", "mean"),
        avg_recommended_occupancy=("recommended_occupancy_rate", "mean"),
        current_revenue=("revenue", "sum"),
        recommended_revenue=("recommended_revenue", "sum"),
        revenue_uplift=("revenue_uplift", "sum"),
        current_margin=("margin", "sum"),
        recommended_margin=("recommended_margin", "sum"),
        margin_uplift=("margin_uplift", "sum"),
        avg_current_margin_pct=("margin_pct", "mean"),
        avg_recommended_margin_pct=("recommended_margin_pct", "mean"),
        avg_elasticity=("elasticity_for_simulation", "mean"),
        avg_score=("recommendation_score", "mean")
    )
    .reset_index()
)

route_recommendation_summary["revenue_uplift_pct"] = np.where(
    route_recommendation_summary["current_revenue"] > 0,
    route_recommendation_summary["revenue_uplift"] / route_recommendation_summary["current_revenue"],
    0
)

route_recommendation_summary["margin_uplift_pct"] = np.where(
    route_recommendation_summary["current_margin"] != 0,
    route_recommendation_summary["margin_uplift"] / route_recommendation_summary["current_margin"],
    0
)

route_recommendation_summary["confidence_score"] = route_recommendation_summary["avg_score"]
route_recommendation_summary["confidence_level"] = route_recommendation_summary["confidence_score"].apply(classify_confidence)

route_recommendation_summary = route_recommendation_summary.sort_values("revenue_uplift_pct", ascending=False)

route_recommendation_summary

In [ ]:
# ============================================================
# 16. ACCIÓN DOMINANTE POR RUTA
# ============================================================

route_action_distribution = (
    best_trip_recommendations
    .groupby(["route", "pricing_action"])
    .size()
    .reset_index(name="number_of_trips")
)

route_action_distribution["route_total_trips"] = route_action_distribution.groupby("route")["number_of_trips"].transform("sum")

route_action_distribution["action_share"] = (
    route_action_distribution["number_of_trips"] /
    route_action_distribution["route_total_trips"]
)

dominant_route_action = (
    route_action_distribution
    .sort_values(["route", "number_of_trips"], ascending=[True, False])
    .groupby("route")
    .head(1)
    .rename(columns={
        "pricing_action": "dominant_pricing_action",
        "number_of_trips": "dominant_action_trips",
        "action_share": "dominant_action_share"
    })
)

route_recommendation_summary = route_recommendation_summary.merge(
    dominant_route_action[[
        "route",
        "dominant_pricing_action",
        "dominant_action_trips",
        "dominant_action_share"
    ]],
    on="route",
    how="left"
)

route_recommendation_summary

## 10. Distribución global de acciones

Analizamos la cantidad de viajes por tipo de recomendación.

In [ ]:
# ============================================================
# 17. DISTRIBUCIÓN DE ACCIONES
# ============================================================

action_distribution = (
    best_trip_recommendations["pricing_action"]
    .value_counts()
    .reset_index()
)

action_distribution.columns = ["pricing_action", "number_of_trips"]

action_distribution["share_of_trips"] = (
    action_distribution["number_of_trips"] /
    action_distribution["number_of_trips"].sum()
)

action_distribution

In [ ]:
# ============================================================
# 18. GRÁFICO: DISTRIBUCIÓN DE ACCIONES
# ============================================================

plt.figure(figsize=(12, 6))

sns.barplot(
    data=action_distribution,
    x="number_of_trips",
    y="pricing_action"
)

plt.title("Distribución global de acciones recomendadas")
plt.xlabel("Número de viajes")
plt.ylabel("Acción recomendada")
plt.tight_layout()

plt.savefig(f"{images_path}/40_pricing_recommendation_action_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

## 11. Visualizaciones ejecutivas por ruta

In [ ]:
# ============================================================
# 19. CAMBIO MEDIO DE PRECIO RECOMENDADO POR RUTA
# ============================================================

plt.figure(figsize=(12, 6))

plot_df = route_recommendation_summary.sort_values("avg_price_change_pct", ascending=False)

sns.barplot(
    data=plot_df,
    x="avg_price_change_pct",
    y="route"
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.title("Cambio medio de precio recomendado por ruta")
plt.xlabel("Cambio medio recomendado")
plt.ylabel("Ruta")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()

plt.savefig(f"{images_path}/41_avg_recommended_price_change_by_route.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# 20. UPLIFT DE REVENUE POR RUTA
# ============================================================

plt.figure(figsize=(12, 6))

plot_df = route_recommendation_summary.sort_values("revenue_uplift_pct", ascending=False)

sns.barplot(
    data=plot_df,
    x="revenue_uplift_pct",
    y="route"
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.title("Uplift estimado de revenue por ruta")
plt.xlabel("Revenue uplift estimado")
plt.ylabel("Ruta")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()

plt.savefig(f"{images_path}/42_revenue_uplift_by_route_recommendation.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# 21. UPLIFT DE MARGEN POR RUTA
# ============================================================

plt.figure(figsize=(12, 6))

plot_df = route_recommendation_summary.sort_values("margin_uplift_pct", ascending=False)

sns.barplot(
    data=plot_df,
    x="margin_uplift_pct",
    y="route"
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.title("Uplift estimado de margen por ruta")
plt.xlabel("Margin uplift estimado")
plt.ylabel("Ruta")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()

plt.savefig(f"{images_path}/43_margin_uplift_by_route_recommendation.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# 22. HEATMAP DE ACCIONES POR RUTA
# ============================================================

action_pivot = pd.crosstab(
    best_trip_recommendations["route"],
    best_trip_recommendations["pricing_action"],
    normalize="index"
)

plt.figure(figsize=(12, 6))

sns.heatmap(
    action_pivot,
    annot=True,
    fmt=".1%",
    cmap="Blues"
)

plt.title("Distribución porcentual de acciones recomendadas por ruta")
plt.xlabel("Acción recomendada")
plt.ylabel("Ruta")
plt.tight_layout()

plt.savefig(f"{images_path}/44_heatmap_pricing_actions_by_route.png", dpi=300, bbox_inches="tight")

plt.show()

## 12. Recomendaciones de alta prioridad y revisión manual

In [ ]:
# ============================================================
# 23. RECOMENDACIONES DE ALTA PRIORIDAD
# ============================================================

high_priority_recommendations = best_trip_recommendations[
    (best_trip_recommendations["confidence_level"] == "High") &
    (best_trip_recommendations["recommendation_score"] >= 0.78) &
    (best_trip_recommendations["revenue_uplift_pct"] > 0) &
    (best_trip_recommendations["margin_uplift_pct"] > 0)
].copy()

high_priority_recommendations = high_priority_recommendations.sort_values(
    "recommendation_score",
    ascending=False
)

high_priority_recommendations.head(20)

In [ ]:
# ============================================================
# 24. CASOS PARA REVISIÓN MANUAL
# ============================================================

manual_review_recommendations = best_trip_recommendations[
    (
        best_trip_recommendations["pricing_action"].isin([
            "Manual review",
            "Margin review",
            "Capacity / premium review"
        ])
    ) |
    (
        best_trip_recommendations["confidence_level"].isin([
            "Low",
            "Manual review"
        ])
    )
].copy()

manual_review_recommendations = manual_review_recommendations.sort_values(
    ["recommendation_score", "margin_uplift_pct"],
    ascending=[True, True]
)

manual_review_recommendations.head(20)

## 13. Simulador interactivo del recomendador

Este bloque permite filtrar recomendaciones por ruta, temporada y acción.

In [ ]:
# ============================================================
# 25. SIMULADOR INTERACTIVO EN COLAB
# ============================================================

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output

    route_options = ["All"] + sorted(best_trip_recommendations["route"].unique())
    season_options = ["All"] + sorted(best_trip_recommendations["season"].unique())
    action_options = ["All"] + sorted(best_trip_recommendations["pricing_action"].unique())

    route_selector = widgets.Dropdown(options=route_options, value="All", description="Ruta:", layout=widgets.Layout(width="420px"))
    season_selector = widgets.Dropdown(options=season_options, value="All", description="Temporada:", layout=widgets.Layout(width="420px"))
    action_selector = widgets.Dropdown(options=action_options, value="All", description="Acción:", layout=widgets.Layout(width="420px"))

    recommender_output = widgets.Output()

    def eur(value):
        return f"{value:,.0f} €".replace(",", "X").replace(".", ",").replace("X", ".")

    def pct_es(value):
        return f"{value:.1%}".replace(".", ",")

    def render_recommender_summary(*args):
        with recommender_output:
            clear_output(wait=True)

            filtered = best_trip_recommendations.copy()

            if route_selector.value != "All":
                filtered = filtered[filtered["route"] == route_selector.value]

            if season_selector.value != "All":
                filtered = filtered[filtered["season"] == season_selector.value]

            if action_selector.value != "All":
                filtered = filtered[filtered["pricing_action"] == action_selector.value]

            if len(filtered) == 0:
                display(HTML("<p>No hay resultados para los filtros seleccionados.</p>"))
                return

            trips = len(filtered)
            avg_price_change = filtered["recommended_price_change_pct"].mean()
            revenue_uplift = filtered["revenue_uplift"].sum()
            margin_uplift = filtered["margin_uplift"].sum()
            avg_score = filtered["recommendation_score"].mean()

            html = f'''
            <div style="font-family:Arial; margin-top:10px;">
                <h3>Resumen del recomendador</h3>
                <div style="display:grid; grid-template-columns: repeat(4, 1fr); gap:10px;">
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Viajes filtrados</div>
                        <div style="font-size:24px; font-weight:bold;">{trips:,}</div>
                    </div>
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Cambio precio medio</div>
                        <div style="font-size:24px; font-weight:bold;">{pct_es(avg_price_change)}</div>
                    </div>
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Revenue uplift</div>
                        <div style="font-size:24px; font-weight:bold;">{eur(revenue_uplift)}</div>
                    </div>
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Score medio</div>
                        <div style="font-size:24px; font-weight:bold;">{avg_score:.2%}</div>
                    </div>
                </div>
            </div>
            '''

            display(HTML(html))

            action_counts = filtered["pricing_action"].value_counts().reset_index()
            action_counts.columns = ["pricing_action", "number_of_trips"]

            plt.figure(figsize=(10, 4))
            sns.barplot(
                data=action_counts,
                x="number_of_trips",
                y="pricing_action"
            )
            plt.title("Acciones recomendadas")
            plt.xlabel("Número de viajes")
            plt.ylabel("Acción")
            plt.tight_layout()
            plt.show()

            display(
                filtered[[
                    "trip_id",
                    "trip_date",
                    "route",
                    "season",
                    "avg_ticket_price",
                    "recommended_ticket_price",
                    "recommended_price_change_pct",
                    "recommended_occupancy_rate",
                    "revenue_uplift_pct",
                    "margin_uplift_pct",
                    "pricing_action",
                    "confidence_level",
                    "recommendation_explanation"
                ]].head(15)
            )

    for widget in [route_selector, season_selector, action_selector]:
        widget.observe(render_recommender_summary, names="value")

    display(
        widgets.VBox([
            widgets.HTML("<h3>Filtros del recomendador</h3>"),
            route_selector,
            season_selector,
            action_selector,
            recommender_output
        ])
    )

    render_recommender_summary()

except Exception as e:
    print("No se pudo cargar el simulador interactivo en este entorno.")
    print("Error:", e)

## 14. Exportación HTML del recomendador

Generamos un explorador HTML autónomo para revisar recomendaciones fuera de Colab.

In [ ]:
# ============================================================
# 26. EXPORTAR HTML DEL RECOMENDADOR
# ============================================================

dashboard_recommendations = best_trip_recommendations.copy()

max_rows_html = 1500

if len(dashboard_recommendations) > max_rows_html:
    dashboard_recommendations_html = dashboard_recommendations.sample(max_rows_html, random_state=42).copy()
else:
    dashboard_recommendations_html = dashboard_recommendations.copy()

html_columns = [
    "trip_id",
    "trip_date",
    "route",
    "season",
    "day_of_week",
    "departure_hour",
    "booking_window",
    "avg_ticket_price",
    "recommended_ticket_price",
    "recommended_price_change_pct",
    "recommended_occupancy_rate",
    "revenue_uplift_pct",
    "margin_uplift_pct",
    "recommendation_score",
    "confidence_level",
    "pricing_action",
    "recommendation_explanation"
]

html_data = dashboard_recommendations_html[html_columns].copy()
html_data["trip_date"] = html_data["trip_date"].astype(str)

recommendations_json = json.dumps(html_data.to_dict(orient="records"), ensure_ascii=False)

html_output = """
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Levante Ferries — Pricing Recommendation Explorer</title>
<style>
body{font-family:Arial,sans-serif;margin:0;padding:24px;background:#f5f7fb;color:#102a43}
.container{max-width:1200px;margin:0 auto}
h1{margin:0 0 8px 0;font-size:34px;letter-spacing:-.04em}
p{color:#627d98}
.panel{background:white;border:1px solid #d9e2ec;border-radius:16px;padding:16px;margin-bottom:16px;box-shadow:0 12px 30px rgba(16,42,67,.06)}
.controls{display:grid;grid-template-columns:repeat(4,1fr);gap:12px}
label{font-size:13px;color:#627d98;font-weight:700}
select{width:100%;margin-top:6px;padding:9px;border:1px solid #d9e2ec;border-radius:9px}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px}
.kpi{background:white;border:1px solid #d9e2ec;border-radius:14px;padding:14px}
.kpi span{display:block;color:#627d98;font-size:13px}
.kpi strong{display:block;font-size:24px;margin-top:6px}
table{width:100%;border-collapse:collapse;font-size:13px}
th,td{border-bottom:1px solid #e5eaf0;padding:8px;text-align:left;vertical-align:top}
th{background:#f3f7fb}
.badge{display:inline-block;padding:4px 8px;border-radius:999px;font-size:12px;font-weight:700}
.increase{background:#d1fae5;color:#065f46}
.promo{background:#dbeafe;color:#1e40af}
.review{background:#fee2e2;color:#991b1b}
.maintain{background:#fef3c7;color:#92400e}
@media(max-width:900px){.controls,.kpis{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="container">
<h1>Pricing Recommendation Explorer</h1>
<p>Explorador interactivo de recomendaciones de pricing por viaje.</p>

<div class="panel controls">
<label>Ruta<select id="routeFilter"></select></label>
<label>Temporada<select id="seasonFilter"></select></label>
<label>Acción<select id="actionFilter"></select></label>
<label>Confianza<select id="confidenceFilter"></select></label>
</div>

<div class="kpis">
<div class="kpi"><span>Viajes filtrados</span><strong id="kTrips">-</strong></div>
<div class="kpi"><span>Cambio precio medio</span><strong id="kPriceChange">-</strong></div>
<div class="kpi"><span>Revenue uplift medio</span><strong id="kRevenue">-</strong></div>
<div class="kpi"><span>Score medio</span><strong id="kScore">-</strong></div>
</div>

<div class="panel">
<table>
<thead>
<tr>
<th>Viaje</th><th>Fecha</th><th>Ruta</th><th>Temporada</th><th>Precio actual</th><th>Precio recomendado</th><th>Cambio</th><th>Acción</th><th>Confianza</th><th>Explicación</th>
</tr>
</thead>
<tbody id="tableBody"></tbody>
</table>
</div>
</div>

<script>
const data = __DATA__;

const euro = v => new Intl.NumberFormat("es-ES",{style:"currency",currency:"EUR",maximumFractionDigits:2}).format(v);
const pct = v => new Intl.NumberFormat("es-ES",{style:"percent",maximumFractionDigits:1,signDisplay:"exceptZero"}).format(v);

function uniqueValues(field){
    return [...new Set(data.map(d => d[field]).filter(Boolean))].sort();
}

function fillSelect(id, values){
    const select = document.getElementById(id);
    select.innerHTML = "";
    select.add(new Option("All", "All"));
    values.forEach(v => select.add(new Option(v, v)));
}

function badgeClass(action){
    if(action === "Increase price") return "increase";
    if(action === "Promotional action") return "promo";
    if(action.includes("review")) return "review";
    return "maintain";
}

function applyFilters(){
    const route = document.getElementById("routeFilter").value;
    const season = document.getElementById("seasonFilter").value;
    const action = document.getElementById("actionFilter").value;
    const confidence = document.getElementById("confidenceFilter").value;

    let filtered = data.slice();

    if(route !== "All") filtered = filtered.filter(d => d.route === route);
    if(season !== "All") filtered = filtered.filter(d => d.season === season);
    if(action !== "All") filtered = filtered.filter(d => d.pricing_action === action);
    if(confidence !== "All") filtered = filtered.filter(d => d.confidence_level === confidence);

    const avg = arr => arr.length ? arr.reduce((a,b)=>a+b,0)/arr.length : 0;

    document.getElementById("kTrips").textContent = filtered.length.toLocaleString("es-ES");
    document.getElementById("kPriceChange").textContent = pct(avg(filtered.map(d => d.recommended_price_change_pct)));
    document.getElementById("kRevenue").textContent = pct(avg(filtered.map(d => d.revenue_uplift_pct)));
    document.getElementById("kScore").textContent = pct(avg(filtered.map(d => d.recommendation_score)));

    const rows = filtered
        .sort((a,b) => b.recommendation_score - a.recommendation_score)
        .slice(0, 50)
        .map(d => `
            <tr>
                <td>${d.trip_id}</td>
                <td>${d.trip_date}</td>
                <td>${d.route}</td>
                <td>${d.season}</td>
                <td>${euro(d.avg_ticket_price)}</td>
                <td>${euro(d.recommended_ticket_price)}</td>
                <td>${pct(d.recommended_price_change_pct)}</td>
                <td><span class="badge ${badgeClass(d.pricing_action)}">${d.pricing_action}</span></td>
                <td>${d.confidence_level}</td>
                <td>${d.recommendation_explanation}</td>
            </tr>
        `)
        .join("");

    document.getElementById("tableBody").innerHTML = rows;
}

fillSelect("routeFilter", uniqueValues("route"));
fillSelect("seasonFilter", uniqueValues("season"));
fillSelect("actionFilter", uniqueValues("pricing_action"));
fillSelect("confidenceFilter", uniqueValues("confidence_level"));

["routeFilter","seasonFilter","actionFilter","confidenceFilter"].forEach(id => {
    document.getElementById(id).addEventListener("change", applyFilters);
});

applyFilters();
</script>
</body>
</html>
"""

html_output = html_output.replace("__DATA__", recommendations_json)

html_path = f"{dashboard_path}/pricing_recommendation_explorer.html"

with open(html_path, "w", encoding="utf-8") as file:
    file.write(html_output)

print("Explorador HTML exportado en:")
print(html_path)

## 15. Insights automáticos

In [ ]:
# ============================================================
# 27. INSIGHTS AUTOMÁTICOS
# ============================================================

top_route_revenue = route_recommendation_summary.sort_values("revenue_uplift_pct", ascending=False).iloc[0]
top_route_margin = route_recommendation_summary.sort_values("margin_uplift_pct", ascending=False).iloc[0]
most_common_action = action_distribution.sort_values("number_of_trips", ascending=False).iloc[0]
highest_confidence_route = route_recommendation_summary.sort_values("confidence_score", ascending=False).iloc[0]

print("INSIGHTS EJECUTIVOS — RECOMENDADOR FORMAL DE PRICING")
print("-" * 80)

print(
    f"1. La acción más frecuente es '{most_common_action['pricing_action']}' "
    f"con {most_common_action['number_of_trips']:,.0f} viajes "
    f"({most_common_action['share_of_trips']:.2%} del total)."
)

print(
    f"2. La mayor oportunidad de revenue aparece en {top_route_revenue['route']} "
    f"con un uplift estimado del {top_route_revenue['revenue_uplift_pct']:.2%}."
)

print(
    f"3. La mayor oportunidad de margen aparece en {top_route_margin['route']} "
    f"con un uplift estimado del {top_route_margin['margin_uplift_pct']:.2%}."
)

print(
    f"4. La ruta con mayor confianza media es {highest_confidence_route['route']} "
    f"con un score medio del {highest_confidence_route['confidence_score']:.2%}."
)

print("\nAcción dominante por ruta:")
for _, row in route_recommendation_summary.sort_values("route").iterrows():
    print(
        f"- {row['route']}: {row['dominant_pricing_action']} "
        f"({row['dominant_action_share']:.2%} de los viajes)"
    )

## 16. Guardado de resultados

In [ ]:
# ============================================================
# 28. GUARDADO DE TABLAS
# ============================================================

tables_to_export = {
    "pricing_recommender_scenarios": recommendation_scenarios,
    "pricing_recommender_trip_recommendations": best_trip_recommendations,
    "pricing_recommender_route_summary": route_recommendation_summary,
    "pricing_recommender_action_distribution": action_distribution,
    "pricing_recommender_route_action_distribution": route_action_distribution,
    "pricing_recommender_high_priority": high_priority_recommendations,
    "pricing_recommender_manual_review": manual_review_recommendations
}

for file_name, table in tables_to_export.items():
    export_table_files(
        dataframe=table,
        output_folder=reports_path,
        file_name=file_name,
        sheet_name=file_name[:31]
    )

print("Tablas del recomendador exportadas correctamente.")

## 17. Resumen ejecutivo en Markdown

In [ ]:
# ============================================================
# 29. RESUMEN EJECUTIVO EN MARKDOWN
# ============================================================

total_recommended_trips = len(best_trip_recommendations)
high_confidence_count = (best_trip_recommendations["confidence_level"] == "High").sum()
manual_review_count = len(manual_review_recommendations)

total_revenue_uplift = best_trip_recommendations["revenue_uplift"].sum()
total_margin_uplift = best_trip_recommendations["margin_uplift"].sum()

executive_summary = f"""
# Executive Summary — Formal Pricing Recommendation Engine

## Project context

This notebook builds a formal pricing recommendation engine for a simulated ferry operator: Levante Ferries.

The engine combines:

- Demand prediction.
- Price-demand elasticity.
- Occupancy.
- Margin.
- Competitive price positioning.
- Scenario simulation.
- Risk flags.
- Explainable recommendation rules.

## Recommendation scope

- Trips analyzed: {total_recommended_trips:,.0f}
- High-confidence recommendations: {high_confidence_count:,.0f}
- Manual review cases: {manual_review_count:,.0f}

## Estimated impact

- Total estimated revenue uplift: {total_revenue_uplift:,.2f} €
- Total estimated margin uplift: {total_margin_uplift:,.2f} €

## Key findings

- Most frequent action: {most_common_action['pricing_action']}
- Top revenue opportunity route: {top_route_revenue['route']} with {top_route_revenue['revenue_uplift_pct']:.2%} uplift
- Top margin opportunity route: {top_route_margin['route']} with {top_route_margin['margin_uplift_pct']:.2%} uplift
- Highest confidence route: {highest_confidence_route['route']} with {highest_confidence_route['confidence_score']:.2%} average score

## Business interpretation

The recommendation engine should be used as a decision-support layer, not as an automatic pricing system.

The most reliable recommendations are those where:

- Demand remains healthy.
- Margin improves.
- Revenue improves.
- Competitive price position remains reasonable.
- Confidence score is high.

Manual review is required when recommendations show low confidence, margin pressure, saturation risk or aggressive competitive positioning.

## Next step

The next notebook will prepare the final executive dashboard, combining KPIs, pricing scenarios, recommendations and route-level insights.
"""

summary_path = f"{reports_path}/executive_summary_notebook_06.md"

with open(summary_path, "w", encoding="utf-8") as file:
    file.write(executive_summary)

print("Resumen ejecutivo guardado en:")
print(summary_path)

In [ ]:
# ============================================================
# 30. ARCHIVOS GENERADOS
# ============================================================

print("Imágenes generadas en el Notebook 6:")
for file in sorted(os.listdir(images_path)):
    if file.endswith(".png") and file.startswith(("40_", "41_", "42_", "43_", "44_")):
        print("-", file)

print("\nReportes generados en el Notebook 6:")
for file in sorted(os.listdir(reports_path)):
    if (
        file.startswith("pricing_recommender_") or
        file == "executive_summary_notebook_06.md"
    ):
        print("-", file)

print("\nDashboard / herramientas HTML:")
for file in sorted(os.listdir(dashboard_path)):
    if file in ["pricing_recommendation_explorer.html"]:
        print("-", file)

## Conclusiones del Notebook 6

En este notebook se ha construido un recomendador formal de pricing.

El sistema ha permitido:

- Combinar elasticidad, demanda, ocupación, margen y competencia.
- Simular escenarios de precio de forma autónoma.
- Calcular un score de recomendación.
- Clasificar la confianza de cada recomendación.
- Asignar acciones de pricing por viaje.
- Generar explicaciones automáticas.
- Resumir recomendaciones por ruta.
- Detectar viajes de alta prioridad.
- Separar casos para revisión manual.
- Exportar tablas, gráficos y un explorador HTML.

La principal conclusión es que el pricing dinámico debe ser explicable.

No basta con decir “subir” o “bajar” precio.  
El sistema debe justificar la recomendación con señales de negocio:

- Demanda.
- Elasticidad.
- Ocupación.
- Margen.
- Competencia.
- Riesgo.
- Confianza.

Este notebook prepara la base para el Notebook 7, donde construiremos el dashboard ejecutivo final.